# Figure 1b — Distribution of Mean Discharge Across Stations
**Target journal:** Computers and Geosciences  
**Figure:** KDE + histogram of log₁₀(Qmean) across the 33 gauging stations in the Ebro River Basin.  
**Output:** `output/figures/Figure_Discharge_Distribution.png` at 300 DPI (single-column, ~90 mm)

In [ ]:
# =============================================================================
# CELL 1 — Imports & global rcParams
# =============================================================================
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import gaussian_kde

# ── Match the global style of the project (serif, clean ticks) ───────────────
mpl.rcParams.update({
    'font.family'        : 'serif',
    'font.serif'         : ['Times New Roman', 'DejaVu Serif', 'serif'],
    'font.size'          : 8,
    'axes.linewidth'     : 0.7,
    'xtick.major.width'  : 0.7,
    'ytick.major.width'  : 0.7,
    'xtick.minor.width'  : 0.4,
    'ytick.minor.width'  : 0.4,
    'xtick.major.size'   : 4,
    'ytick.major.size'   : 4,
    'xtick.minor.size'   : 2,
    'ytick.minor.size'   : 2,
    'xtick.direction'    : 'out',
    'ytick.direction'    : 'out',
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'figure.dpi'         : 150,     # screen preview
    'savefig.dpi'        : 300,     # publication output
})

print('Libraries loaded.')

In [ ]:
# =============================================================================
# CELL 2 — File paths
# =============================================================================
CSV_PATH = os.path.join('data', 'SSI_daily.csv')
OUT_PATH = os.path.join('output', 'figures', 'Figure_Discharge_Distribution.png')

print(f'Input  : {CSV_PATH}')
print(f'Output : {OUT_PATH}')

In [ ]:
# =============================================================================
# CELL 3 — Load data & compute Qmean per station
# =============================================================================
# Load the full daily discharge dataset
df = pd.read_csv(CSV_PATH, parse_dates=['date'])
print(f'Dataset shape   : {df.shape}')
print(f'Stations        : {df["station_id"].nunique()}')
print(f'Date range      : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Q zeros/negatives: {(df["Q"] <= 0).sum()} rows')

# ── Safe log transform: discard Q ≤ 0 before aggregating ─────────────────────
# Zeros represent dry-channel or no-data conditions; they would make the
# mean artificially low and cannot be log-transformed. We exclude them
# from the per-station mean so the result reflects typical flow, not
# the proportion of zero-flow days.
df_valid = df[df['Q'] > 0].copy()

# Mean discharge per station (m³ s⁻¹)
qmean = (
    df_valid
    .groupby('station_id')['Q']
    .mean()
    .rename('Qmean')
    .reset_index()
)

# Log10 transform
qmean['log10_Qmean'] = np.log10(qmean['Qmean'])

print(f'\nStations with valid Qmean : {len(qmean)}')
print(f'Qmean range  : {qmean["Qmean"].min():.2f} – {qmean["Qmean"].max():.2f} m³ s⁻¹')
print(f'log10 range  : {qmean["log10_Qmean"].min():.2f} – {qmean["log10_Qmean"].max():.2f}')
print('\nFull Qmean summary:')
print(qmean[['Qmean', 'log10_Qmean']].describe().round(3))

In [ ]:
# =============================================================================
# CELL 4 — Compute KDE and summary statistics
# =============================================================================
x_vals = qmean['log10_Qmean'].values   # shape (N,)
n_stations = len(x_vals)

# ── KDE using scipy (Scott bandwidth rule) ────────────────────────────────────
kde = gaussian_kde(x_vals, bw_method='scott')

# Evaluation grid: span the data range with padding
x_pad  = 0.4
x_grid = np.linspace(x_vals.min() - x_pad, x_vals.max() + x_pad, 500)
y_grid = kde(x_grid)

# ── Summary statistics ────────────────────────────────────────────────────────
x_median = np.median(x_vals)
x_mean   = np.mean(x_vals)
x_min    = x_vals.min()
x_max    = x_vals.max()
x_p25    = np.percentile(x_vals, 25)
x_p75    = np.percentile(x_vals, 75)

# Identify stations at the extremes for annotation
idx_min = qmean['log10_Qmean'].idxmin()
idx_max = qmean['log10_Qmean'].idxmax()
station_min = qmean.loc[idx_min, 'station_id']
station_max = qmean.loc[idx_max, 'station_id']
qmean_min   = qmean.loc[idx_min, 'Qmean']
qmean_max   = qmean.loc[idx_max, 'Qmean']

print(f'n stations : {n_stations}')
print(f'Median     : {x_median:.3f}  ({10**x_median:.1f} m³ s⁻¹)')
print(f'Mean       : {x_mean:.3f}    ({10**x_mean:.1f} m³ s⁻¹)')
print(f'Min        : {x_min:.3f}     (station {station_min}, {qmean_min:.2f} m³ s⁻¹)')
print(f'Max        : {x_max:.3f}     (station {station_max}, {qmean_max:.1f} m³ s⁻¹)')

In [ ]:
# =============================================================================
# CELL 5 — Build the figure
# =============================================================================
# Figure sized to fit a single journal column (≈ 88 mm ≈ 3.46 in)
# Aspect ratio 1:1.1 (slightly taller than wide for a compact panel)
FIG_W = 3.46    # inches
FIG_H = 3.50    # inches

# ── Colour palette (monochrome / grey tones only) ─────────────────────────────
C_HIST    = '#7ab5d4'   # muted steel blue  – histogram bars
C_HIST_ED = '#3a85b0'   # medium blue       – histogram bar edges
C_KDE     = '#1a4f72'   # deep navy         – KDE line
C_FILL    = '#c5dff3'   # very light blue   – KDE fill
C_MEDIAN  = '#ca6f1e'   # warm amber/orange – median line (contrasting accent)
C_IQR     = '#aed6f1'   # sky blue          – IQR band
C_RUG     = '#2e86c1'   # medium blue       – rug ticks
C_ANNOT   = '#1b2631'   # near-black navy   – annotation text

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))

# ── Determine histogram bin edges (Freedman–Diaconis heuristic) ──────────────
# For n=33 stations, using ~6 bins looks clean; fd rule is used as a guide
iqr_vals = x_p75 - x_p25
bin_width_fd = 2 * iqr_vals / (n_stations ** (1/3))   # Freedman–Diaconis
# Enforce a minimum of 5 and maximum of 8 bins for visual clarity at n=33
n_bins = max(5, min(8, int(np.ceil((x_max - x_min) / bin_width_fd))))
print(f'Using {n_bins} histogram bins (bin width ≈ {(x_max - x_min)/n_bins:.2f} log units)')

# ── 1. Histogram ─────────────────────────────────────────────────────────────
counts, bin_edges, patches = ax.hist(
    x_vals,
    bins      = n_bins,
    density   = True,          # normalise to density so it is comparable to KDE
    color     = C_HIST,
    edgecolor = C_HIST_ED,
    linewidth = 0.6,
    alpha     = 0.70,
    zorder    = 2,
    label     = 'Histogram'
)

# ── 2. IQR shaded band ───────────────────────────────────────────────────────
ax.axvspan(x_p25, x_p75, color=C_IQR, alpha=0.30, zorder=1,
           label=f'IQR [{10**x_p25:.0f}–{10**x_p75:.0f} m³ s⁻¹]')

# ── 3. KDE fill ──────────────────────────────────────────────────────────────
ax.fill_between(x_grid, y_grid,
                color=C_FILL, alpha=0.45, zorder=3)

# ── 4. KDE line ──────────────────────────────────────────────────────────────
ax.plot(x_grid, y_grid,
        color=C_KDE, linewidth=1.5, zorder=4,
        label='KDE (Scott bandwidth)')

# ── 5. Median vertical line ───────────────────────────────────────────────────
ax.axvline(x_median,
           color=C_MEDIAN, linewidth=1.2, linestyle='--', zorder=5,
           label=f'Median ({10**x_median:.1f} m³ s⁻¹)')

# ── 6. Rug plot: one tick per station along x-axis ───────────────────────────
rug_y_top = ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1.0
# We draw the rug after finalising the y-limits; use a transform trick
rug_height = 0.030   # fraction of axes height
for xv in x_vals:
    ax.plot(
        [xv, xv],
        [0, rug_height],                      # in axes-fraction coordinates
        transform=ax.get_xaxis_transform(),   # x=data, y=axes fraction
        color=C_RUG, linewidth=0.8, alpha=0.9, zorder=6, clip_on=True
    )

# ── 7. Min / Max annotations ─────────────────────────────────────────────────
# Min station annotation — left side
ax.annotate(
    f'{qmean_min:.2f}\nm³ s⁻¹',
    xy=(x_min, 0),
    xytext=(x_min + 0.05, ax.get_ylim()[1] * 0.60 if ax.get_ylim()[1] > 0 else 0.6),
    xycoords='data', textcoords='data',
    fontsize=6.5, ha='left', va='top', color=C_ANNOT,
    arrowprops=dict(arrowstyle='->', color=C_ANNOT, lw=0.7),
    zorder=7
)
# Max station annotation — right side
ax.annotate(
    f'{qmean_max:.0f}\nm³ s⁻¹',
    xy=(x_max, 0),
    xytext=(x_max - 0.05, ax.get_ylim()[1] * 0.60 if ax.get_ylim()[1] > 0 else 0.6),
    xycoords='data', textcoords='data',
    fontsize=6.5, ha='right', va='top', color=C_ANNOT,
    arrowprops=dict(arrowstyle='->', color=C_ANNOT, lw=0.7),
    zorder=7
)

# ── 8. Axes labels & formatting ───────────────────────────────────────────────
ax.set_xlabel(r'Mean discharge  $\log_{10}$ (m³ s⁻¹)', fontsize=8)
ax.set_ylabel('Density', fontsize=8)

# Replace x-tick labels with both log10 value AND the original m³/s value
# so readers can quickly cross-reference with the map
x_tick_log = np.arange(
    np.floor(x_min - 0.1),
    np.ceil(x_max + 0.2) + 0.5,
    0.5
)
ax.set_xticks(x_tick_log)
ax.set_xticklabels(
    [f'{v:.1f}' for v in x_tick_log],
    fontsize=7
)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.tick_params(axis='both', which='major', labelsize=7)
ax.tick_params(axis='x', which='minor', bottom=True)
ax.xaxis.set_minor_locator(mticker.MultipleLocator(0.25))

# Tighten x-limits to data range with small padding
ax.set_xlim(x_vals.min() - x_pad * 0.8, x_vals.max() + x_pad * 0.8)

# ── 9. Secondary x-axis showing raw m³/s values ──────────────────────────────
# This helps readers who are more comfortable with natural units
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())

# Pick a few representative tick positions in log10 space
raw_ticks_m3s = [0.5, 1, 5, 10, 50, 100, 500, 1000]   # m³/s values
log_ticks     = [np.log10(v) for v in raw_ticks_m3s
                 if ax.get_xlim()[0] <= np.log10(v) <= ax.get_xlim()[1]]
raw_labels    = [str(int(v)) if v >= 1 else str(v)
                 for v in raw_ticks_m3s
                 if ax.get_xlim()[0] <= np.log10(v) <= ax.get_xlim()[1]]

ax2.set_xticks(log_ticks)
ax2.set_xticklabels(raw_labels, fontsize=6.5)
ax2.set_xlabel(r'Mean discharge  (m³ s⁻¹)', fontsize=7, labelpad=4)
ax2.tick_params(axis='x', which='major', length=3, width=0.6, labelsize=6.5)

# ── 10. Legend ────────────────────────────────────────────────────────────────
leg = ax.legend(
    loc          = 'upper right',
    bbox_to_anchor = (1.05, 1.0),
    fontsize     = 6,
    framealpha   = 0.9,
    edgecolor    = '#888888',
    fancybox     = False,
    borderpad    = 0.6,
    handlelength = 1.4,
)
leg.get_frame().set_linewidth(0.5)

# ── 11. Station count annotation (lower-right) ────────────────────────────────
ax.text(
    0.97, 0.05,
    f'n = {n_stations} stations',
    transform=ax.transAxes,
    ha='right', va='bottom',
    fontsize=6.5, color='#444444',
    style='italic'
)

fig.tight_layout()
plt.show()
print('Figure rendered.')

In [ ]:
# =============================================================================
# CELL 6 — Export at 300 DPI
# =============================================================================
# NOTE: we re-draw min/max annotations here AFTER the axes y-limits have been
# finalised (tight_layout may have changed them slightly in the preview above).
# The cell is self-contained so you can run it independently after tweaking.

fig.savefig(
    OUT_PATH,
    dpi         = 300,
    bbox_inches = 'tight',
    pad_inches  = 0.05,
    facecolor   = 'white',
    transparent = False,
)

print(f'Saved → {OUT_PATH}')

# Quick sanity check: print physical size
from PIL import Image
img = Image.open(OUT_PATH)
w_mm = img.size[0] / 300 * 25.4
h_mm = img.size[1] / 300 * 25.4
print(f'Size : {img.size[0]} × {img.size[1]} px  →  {w_mm:.1f} × {h_mm:.1f} mm at 300 DPI')
img.close()

In [ ]:
# =============================================================================
# CELL 7 — (Optional) Print per-station Qmean table for the paper
# =============================================================================
print(qmean
      .sort_values('Qmean')
      .assign(Qmean=lambda d: d['Qmean'].round(2),
              log10_Qmean=lambda d: d['log10_Qmean'].round(3))
      .to_string(index=False)
)